(4) Orchestrator – Worker

In the orchestrator-workers workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

When to use this workflow:
This workflow is well-suited for complex tasks where you can't predict the subtasks needed (in coding, for example, the number of files that need to be changed and the nature of the change in each file likely depend on the task). Whereas it's topographically similar, the key difference from parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific input.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen-2.5-32b")
result=llm.invoke("Write a short story about a robot learning to love.")

In [ ]:
from typing import Annotated, List
import operator
from typing_extensions import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict

In [ ]:
# Schema for structured output to use in planning
class Section(BaseModel):
    name: str = Field(description="Name for this section of the report")
    description: str = Field(
        description="Brief Overview of the main topics and concepts of the section"
    )


class Sections(BaseModel):
    sections: List[Section] = Field(
        description="Sections of the report"
    )

planner=llm.with_structured_output(Sections)

Creating Workers Dynamically In LangGraph

Because orchestrator-worker workflows are common, LangGraph has the Send API to support this. It lets you dynamically create worker nodes and send each one a specific input. Each worker has its own state, and all worker outputs are written to a shared state key that is accessible to the orchestrator graph. This gives the orchestrator access to all worker output and allows it to synthesize them into a final output. As you can see below, we iterate over a list of sections and Send each to a worker node.

In [ ]:
# Schema for structured output to use in planning
class Section(BaseModel):
    name: str = Field(description="Name for this section of the report")
    description: str = Field(description="Brief Overview of the main topics and concepts of the section")


class Sections(BaseModel):
    sections: List[Section] = Field(
        description="Sections of the report"
    )


# Augment the LLM with schema for structured output
planner = llm.with_structured_output(Sections)

Creating Workers Dynamically In LangGraph

Because orchestrator-worker workflows are common, LangGraph has the Send API to support this. It lets you dynamically create worker nodes and send each one a specific input. Each worker has its own state, and all worker outputs are written to a shared state key that is accessible to the orchestrator graph. This gives the orchestrator access to all worker output and allows it to synthesize them into a final output. As you can see below, we iterate over a list of sections and Send each to a worker node.

In [ ]:
# Graph state

class State(TypedDict):
    topic: str  # Report topic
    sections: list[Section]  # List of report sections

    completed_sections: Annotated[
        list, operator.add
    ]  # All workers write to this key in parallel

    final_report: str  # Final report

Ye runtime par decide karta hai ki kis node ko kitni baar execute karna hai aur kis input ke saath execute karna hai.

Example:

from langgraph.types import Send

def router(state):
    return [
        Send("worker", {"pdf": "a.pdf"}),
        Send("worker", {"pdf": "b.pdf"}),
        Send("worker", {"pdf": "c.pdf"}),
    ]

Yahan router bol raha hai:

"Worker node ko 3 baar chalao."

Result:

          Router
             │
   ┌─────────┼─────────┐
   ▼         ▼         ▼
Worker(A) Worker(B) Worker(C)
   │         │         │
   └─────────┼─────────┘
             ▼
          Aggregator

Ek hi worker node hai, lekin uski 3 parallel executions ho gayi.

#creating workers dynamically in langgraph

In [ ]:
from langgraph.constants import Send

class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

In [ ]:
def orchestrator(state: State):
    """Orchestrator that generates a plan for the report."""

    # Generate report sections using the planner
    report_sections = planner.invoke(
        [
            SystemMessage(
                content="Generate a plan for the report."
            ),
            HumanMessage(
                content=f"Here is the report topic: {state['topic']}"
            ),
        ]
    )

    print("Report Sections:", report_sections)

    return {
        "sections": report_sections.sections
    }

In [ ]:
def llm_call(state: WorkerState):
    """Worker writes a section of the report"""

    # Generate section
    section = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Write a report section following the provided "
                    "name and description. Include no preamble for each section."
                )
            ),
            HumanMessage(
                content=(
                    f"Here is the section name: {state['section'].name} "
                    f"and description: {state['section'].description}"
                )
            ),
        ]
    )

    # Write the generated section to completed_sections
    return {
        "completed_sections": [section.content]
    }

def assign_workers(state: State):
    """Assign a worker to each section in the plan"""

    return [
        Send("llm_call", {"section": s})
        for s in state["sections"]
    ]

def synthesizer(state: State):
    """Synthesize full report from sections"""

    # List of completed sections
    completed_sections = state["completed_sections"]

    # Format completed sections to str to use as context for final sections
    completed_report_sections = "\n\n---\n\n".join(completed_sections)

    return {
        "final_report": completed_report_sections
    }

In [ ]:
# Build workflow
from langgraph.graph import StateGraph, START, END
orchestrator_worker_builder = StateGraph(State)

# Add the nodes
orchestrator_worker_builder.add_node("orchestrator", orchestrator)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesizer)

# Add edges to connect nodes
orchestrator_worker_builder.add_edge(START, "orchestrator")

orchestrator_worker_builder.add_conditional_edges(
    "orchestrator",
    assign_workers,
    ["llm_call"]
)

orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)

StateGraph() 
Main ek graph bana raha hu jisme ek shared state nodes ke beech flow karegi.

dono ka purpose alag hai.
1. Send("llm_call", {...}) → Runtime instruction

Ye actually worker launch karta hai.

Send(
    "llm_call",
    {
        "section": Intro
    }
)

Iska matlab:

"LangGraph, llm_call node execute karo aur is state ke saath execute karo."

Ye actual execution hai.

2. ["llm_call"] → Graph declaration / validation

Ye builder ko pehle hi bata deta hai:

"Mera assign_workers function sirf llm_call node ko hi return karega."

Ye execution nahi karta.

Sirf graph ko validate karta hai.